In [1]:
# ===== A-PATCH: one-hot-count baseline for A2, and a look at layers 10-15 =====
# Self-contained: reloads the cache, so it works in a fresh notebook with the same dataset attached (CPU, ~1-2 min).
import os, glob, numpy as np, pandas as pd, warnings
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
FOLDS, N_BOOT = 5, 300

ROWS = pd.read_csv(glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)[0])
HS = np.concatenate([np.load(f, mmap_mode="r") for f in sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))])
w = ROWS.pivot(index="t", columns="order", values="delta")
M = ROWS[ROWS.order == "none"].set_index("t").loc[w.index][["side", "contrast", "p_right", "n_hist", "n_hist_right"]].copy()
M["shift"] = (w["hist"] - w["none"]).values
pr = M.p_right.round(2).values; blk = np.concatenate([[0], np.cumsum(pr[1:] != pr[:-1])])
m = (M.n_hist.values == 10)
frac = (M.n_hist_right.values + .5) / (M.n_hist.values + 1); TP = np.log(frac / (1 - frac))
cell = pd.get_dummies(M.side.astype(str) + "|" + M.contrast.astype(str), dtype=float).values
C, g, y, tp = cell[m], blk[m], M["shift"].values[m], TP[m]
nr_i = M.n_hist_right.values[m].astype(int)
ONEHOT = pd.get_dummies(nr_i, dtype=float).values                 # free value per count 0..10 (no assumption about the shape)
lsq = lambda A, B: np.linalg.lstsq(A, B, rcond=None)[0]

def oof_z(X):
    z = np.full(len(X), np.nan)
    for tr, te in GroupKFold(FOLDS).split(X, groups=g):
        b = lsq(C[tr], X[tr]); Xtr, Xte = X[tr] - C[tr] @ b, X[te] - C[te] @ b
        yt = tp[tr] - C[tr] @ lsq(C[tr], tp[tr]); d = Xtr.T @ yt; d /= np.linalg.norm(d) + 1e-12; z[te] = Xte @ d
    return z

def pcorr(a, b, Z): return float(np.corrcoef(a - Z @ lsq(Z, a), b - Z @ lsq(Z, b))[0, 1])
def cvr2(yv, Z):
    p = np.zeros_like(yv)
    for tr, te in GroupKFold(FOLDS).split(Z, groups=g): p[te] = Z[te] @ lsq(Z[tr], yv[tr])
    return 1 - ((yv - p) ** 2).sum() / ((yv - yv.mean()) ** 2).sum()
def boot(fn, n=150, seed=0):
    rng = np.random.default_rng(seed); ug = np.unique(g); ib = {k: np.where(g == k)[0] for k in ug}; v = []
    for _ in range(n):
        idx = np.concatenate([ib[k] for k in rng.choice(ug, len(ug))])
        try: v.append(fn(idx))
        except Exception: pass
    return np.nanpercentile(v, [2.5, 97.5])

Zlin, Zoh = np.c_[C, tp], np.c_[C, ONEHOT]                         # cells + smooth count  vs  cells + one free value per count
print("A2 again. partial corr of z with the shift, after removing: (a) cells + smooth count (old), (b) cells + one free value per count (new)")
rows = []
for L in range(6, 33):
    z = oof_z(HS[:, 1, L, :][m].astype(np.float32))
    lo, hi = boot(lambda idx: pcorr(z[idx], y[idx], Zoh[idx]))
    rows.append(dict(layer=L, old=pcorr(z, y, Zlin), new=pcorr(z, y, Zoh), new_lo=lo, new_hi=hi, dR2_new=cvr2(y, np.c_[Zoh, z]) - cvr2(y, Zoh)))
P = pd.DataFrame(rows); print(P.round(3).to_string(index=False))
print("\nlayers where the NEW interval excludes 0:", P.layer[(P.new_lo > 0) | (P.new_hi < 0)].tolist())

print("\nLayers 10-15: is a few-dimension outlier driving things? (norms, variance share of the top 3 dims, evidence separation with them removed)")
hi_c = (M.contrast.values >= 0.09); sr = (M.side.values == "right"); rr = []
for L in range(10, 16):
    Xn, Xh = HS[:, 0, L, :].astype(np.float32), HS[:, 1, L, :].astype(np.float32)
    var = Xh.var(0); top = np.argsort(var)[::-1][:3]; keep = np.ones(Xh.shape[1], bool); keep[top] = False
    def ev(X, mask): return np.linalg.norm(X[hi_c & sr][:, mask].mean(0) - X[hi_c & ~sr][:, mask].mean(0))
    rr.append(dict(layer=L, mean_norm_none=np.linalg.norm(Xn, axis=1).mean(), mean_norm_hist=np.linalg.norm(Xh, axis=1).mean(),
                   top3_dims=str(top.tolist()), top3_var_share=var[top].sum() / var.sum(),
                   evid_ratio_all=ev(Xh, np.ones_like(keep)) / ev(Xn, np.ones_like(keep)), evid_ratio_without_top3=ev(Xh, keep) / ev(Xn, keep)))
print(pd.DataFrame(rr).round(2).to_string(index=False))

A2 again. partial corr of z with the shift, after removing: (a) cells + smooth count (old), (b) cells + one free value per count (new)
 layer    old    new  new_lo  new_hi  dR2_new
     6 -0.142  0.005  -0.074   0.075   -0.001
     7 -0.215  0.016  -0.104   0.096   -0.001
     8 -0.192  0.005  -0.127   0.089   -0.001
     9 -0.265  0.004  -0.125   0.101   -0.002
    10 -0.271 -0.021  -0.186   0.105   -0.003
    11 -0.402 -0.048  -0.208   0.078   -0.004
    12 -0.465 -0.016  -0.172   0.091   -0.003
    13 -0.306  0.007  -0.160   0.120   -0.004
    14 -0.345 -0.078  -0.223   0.037   -0.003
    15 -0.276 -0.113  -0.249  -0.005   -0.001
    16  0.126 -0.114  -0.264  -0.003   -0.002
    17  0.321 -0.091  -0.248   0.029   -0.002
    18  0.256 -0.081  -0.235   0.038   -0.002
    19  0.252 -0.055  -0.218   0.061   -0.002
    20  0.256 -0.059  -0.225   0.058   -0.002
    21  0.306 -0.068  -0.229   0.051   -0.002
    22  0.289 -0.066  -0.225   0.053   -0.002
    23  0.276 -0.067  -0.225   0.053 

How to read it

Part 1 (old vs new partial correlation).

new removes the count with a free value per count 0–10, so a step-shaped behavior can no longer leak into the leftover.
If the negative values at layers 8–15 shrink toward 0, they were an artifact of the smooth-count baseline.
If they stay negative, there is something real at those layers, and I’d want to look at it before steering.
If the positive values at layers 18–32 survive, the “direction predicts behavior beyond the count” claim holds. If they vanish, they were the same step-shape artifact, and the claim goes.
Expect a smaller effect either way, since the baseline gets stronger.

Part 2 (layers 10–15).

The columns to look at are top3_var_share and mean_norm. A layer where 3 dimensions carry most of the variance, or where the norm is far larger than its neighbors, has an outlier dimension. That would explain layer 12’s odd values.
Then compare evid_ratio_all with evid_ratio_without_top3. If layer 12 looks off with the dimensions in and normal without, treat layer 12 as a measurement problem and exclude it from the story.

Paste the output back and I’ll decide the patching layer range and write notebook B.

I need to take back one of my claims from the first A readout. After the stricter baseline, the count direction does not predict the trial shift beyond the count.

## What the patch showed

**1. The "predicts behavior beyond the count" result is gone.**
- With one free value per count (0–10) as the baseline, the partial correlation is about 0 at every layer. Only layers 15–16 are marginally negative (−0.11, interval just clear of 0), and with 27 layers tested I wouldn't read anything into that.
- The earlier +0.26 to +0.34 at layers 18–32 came from the step shape of behavior, which my smooth-count baseline couldn't absorb.
- I said that result "passes the gate". It didn't. The negative values at layers 8–15 were the same artifact.

**2. This is a null, but it can't show the direction isn't used.**
- Once cells and count are removed, little shift variance is left, and the test has little power.
- A correlation can't show whether something is used in any case. That needs an intervention, which is what B is for.

**3. Layer 12 is not an outlier-dimension problem.**
- Norms rise smoothly (17.1 → 18.7 → 20.7), and the top 3 dimensions hold only 1% of the variance. Removing them changes nothing.
- So the 1.79 evidence ratio is a real value of that statistic, not a glitch. My guess at the cause is a confound: side and count are correlated through the blocks, so the right-minus-left difference in history runs also contains the count signal while the count is emerging (layers 11–13). I haven't checked this.

## What stands now

- **Solid:** the count is a linear, early (layers 8–16), faithful readout. A1 and its no-history control are unaffected by the patch.
- **Probably solid:** the representation is mostly linear in the count while behavior is a step, so the step comes later. This was computed without the flawed baseline.
- **Still true, but the onset is uncertain:** evidence separation is about half the no-history value from layer 14 on (0.53, 0.43, then 0.5 later). Where it starts is unclear because of the confound above.
- **Retracted:** "the count direction tracks per-trial behavior".

I updated the checklist with these results.

## Next

1. **A small CPU cell (about 1 minute):** recompute the evidence ratio within each count value (or after removing the count), so the onset layer is clean. This settles whether the suppression starts where the count arrives.
2. **Notebook B (GPU):** steering with the saved prior and shift directions against random and shuffled controls, and whole-residual patching, restricted to layers 8–20. This is where "used" gets decided, so it carries more weight now.
3. **D:** waiting on your account setup. It's independent of everything above.

Shall I write the matched-evidence cell and B together? I'd send them in one message.

In [2]:
# ===== A-PATCH 2: evidence separation measured WITHIN each count value (removes the side/count confound) =====
# Self-contained CPU cell (~1-2 min). Same dataset attached as for notebook A.
import glob, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
N_BOOT = 100

ROWS = pd.read_csv(glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)[0])
HS = np.concatenate([np.load(f, mmap_mode="r") for f in sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))])
M = ROWS[ROWS.order == "none"].sort_values("t").reset_index(drop=True)
nr = M.n_hist_right.values.astype(int); full = (M.n_hist.values == 10)
use = np.where(full & (M.contrast.values >= 0.09))[0]                      # same trials in both runs
side_r = (M.side.values[use] == "right"); cnt = nr[use]
print(f"trials used: {len(use)} (right {side_r.sum()}, left {(~side_r).sum()}); counts with both sides >=3 trials:",
      [c for c in range(11) if ((cnt == c) & side_r).sum() >= 3 and ((cnt == c) & ~side_r).sum() >= 3])

def matched_sep(X, idx_r, idx_l, cnts, rng_idx=None):
    """Right-minus-left mean difference computed within each count value, averaged with weights nR*nL/(nR+nL)."""
    num, den = 0, 0
    for c in range(11):
        r, l = (cnts == c) & idx_r, (cnts == c) & idx_l
        if r.sum() < 3 or l.sum() < 3: continue
        wgt = r.sum() * l.sum() / (r.sum() + l.sum()); num = num + wgt * (X[r].mean(0) - X[l].mean(0)); den += wgt
    return num / den if den else None

rows = []
rng = np.random.default_rng(0)
for L in range(1, 33):
    Xn = HS[use, 0, L, :].astype(np.float32); Xh = HS[use, 1, L, :].astype(np.float32)
    en, eh = matched_sep(Xn, side_r, ~side_r, cnt), matched_sep(Xh, side_r, ~side_r, cnt)
    un = en / (np.linalg.norm(en) + 1e-12)
    rec = dict(layer=L, norm_ratio=np.linalg.norm(eh) / np.linalg.norm(en), axis_ratio=float(eh @ un) / float(en @ un))
    bs = []
    for _ in range(N_BOOT):
        b = rng.integers(0, len(use), len(use)); xn, xh, sr_, cn = Xn[b], Xh[b], side_r[b], cnt[b]
        a, h = matched_sep(xn, sr_, ~sr_, cn), matched_sep(xh, sr_, ~sr_, cn)
        if a is None or h is None: continue
        bs.append(float(h @ (a / (np.linalg.norm(a) + 1e-12))) / float(a @ (a / (np.linalg.norm(a) + 1e-12))))
    rec["axis_lo"], rec["axis_hi"] = np.percentile(bs, [2.5, 97.5]) if bs else (np.nan, np.nan)
    rows.append(rec)
R = pd.DataFrame(rows)
print("\nEvidence signal in the history run relative to the no-history run, measured within each count value")
print("axis_ratio = projection on the no-history evidence axis (1 = no suppression); behavior swing ratio was ~0.55")
print(R.round(2).to_string(index=False))

trials used: 287 (right 134, left 153); counts with both sides >=3 trials: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Evidence signal in the history run relative to the no-history run, measured within each count value
axis_ratio = projection on the no-history evidence axis (1 = no suppression); behavior swing ratio was ~0.55
 layer  norm_ratio  axis_ratio  axis_lo  axis_hi
     1        0.75        0.72     0.66     0.79
     2        0.83        0.76     0.72     0.79
     3        0.88        0.79     0.72     0.83
     4        0.90        0.79     0.72     0.84
     5        0.82        0.69     0.64     0.77
     6        0.79        0.62     0.57     0.69
     7        0.79        0.58     0.54     0.66
     8        0.81        0.54     0.48     0.64
     9        0.82        0.50     0.45     0.59
    10        0.74        0.37     0.33     0.44
    11        0.77        0.38     0.36     0.43
    12        0.83        0.33     0.26     0.44
    13        0.41        0.29     0.27    

axis_ratio is the evidence separation in the history run relative to the no-history run, 1 meaning no suppression. The behavioral swing ratio was about 0.55. The question is the first layer where it drops near that, and whether it still jumps around layers 11–13. If layer 12 stops being an outlier, the confound was the cause and the suppression onset is where the ratio first falls.

What the output means

The strongest result is that the suppression is much bigger than I told you, it starts much earlier, and it has little to do with the prior.

1. The earlier numbers were confounded, as I suspected.

Measured within each count value, the evidence signal in the history run is about 0.3–0.4 of the no-history run from layer 10 onward. My earlier 0.53 was too high.
The layer-12 outlier (1.79) is gone. It came from side and count being correlated, so the count signal leaked into the right-minus-left difference.

2. It is gradual and starts at layer 1, not at layers 12–14.

axis_ratio is 0.72–0.79 at layers 1–4, then falls from 0.69 at layer 5 to 0.37 at layer 10, and stays about 0.3–0.4 after that.
At layers 6–9 the count direction barely exists yet (correlation 0.08 at layer 6, 0.32 at layer 8), yet the evidence is already down to 0.5–0.6. So the count arriving can’t be what suppresses the evidence.
My earlier claim that the suppression begins where the count arrives is retracted.

3. It doesn’t depend on the count at all. I checked this in the behavior data we already have, with no new run. Swing is the right-minus-left Δ at contrast ≥ 0.09, history run over no-history run:

count in prompt	swing ratio
≤2 right	0.50
3–4	0.48
5 (balanced)	0.42
6–7	0.46
≥8	0.40

A balanced 5-of-10 sentence carries no prior and still cuts the swing by more than half. So the evidence loss is a cost of having that sentence in the prompt, not a conflict between prior and evidence. If anything it is slightly worse at extreme counts (0.40 vs 0.42–0.50), but the intervals overlap, so I wouldn’t read anything into that.

4. The residual-stream ratio (0.3–0.4) is lower than the behavioral swing ratio (0.4–0.5). The logit readout is probably nonlinear or saturating, so the two shouldn’t be equated one-to-one.

What this changes
“Prior vs evidence interference” is not a clean finding. It is a format cost, probably extra tokens, longer distance, or diluted attention to the image. The prior effect (+0.75 logits) is a separate, smaller thing sitting on top.
For the mouse comparison, $\gamma/\beta$ computed inside the history condition (12% contrast shift, β = 6.3) is the one that matches how the prior and evidence are actually combined. The no-history β (5%) is the number that mixes two different prompts. I’d report 12% as primary and 5% as a bound, with the format cost stated.
Test C in B (does the prior direction cut the swing) should now come out null. That wouldn’t be a failure; it just agrees with this result.
What to do now

I added a new cell to notebook B (re-attached, replacing the earlier one): B0, “where does the evidence loss come from?” It runs in about 5 minutes and measures the swing under five prompts: no history, a balanced 5-of-10 sentence, a short “Recent trials: 5 left, 5 right.”, a neutral sentence of similar length with no trial information, and the count sentence placed before the image. That separates “any extra text hurts” from “this wording hurts” from “position hurts”, and tells us whether a prompt can carry the prior without the cost.

Run B with QUICK = True first (about 5 min). If the self-tests pass, rerun with QUICK = False. The B0 table appears first in the output.
Paste back everything it prints.
D is still waiting on your IBL account.

If a shorter or reordered format keeps the prior effect and avoids the loss, we’d redo the main run with it and the comparison with the mouse gets cleaner. If nothing does, we report the cost and carry on with the 12% number.